# HazardNet Dual Track Implementation

In [ ]:
import pandas as pd
import numpy as np
import rasterio
import os
import glob
import requests
import time
import warnings
from tqdm import tqdm
from datetime import datetime, timedelta

# Suppress benign rasterio/numpy warnings for cleaner execution logs
warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

# ==============================================================================
# 1. CONFIGURATION & BAND MAPPING
# ==============================================================================
BANDS = {
    'SAR_VV': 0, 
    'SAR_VH': 1, 
    'Blue': 2, 
    'Red': 3,       
    'NIR': 4,       
    'SWIR': 5,      
    'TEMP_2M': 6, 
    'PRECIP': 7, 
    'MAX_TEMP': 8, 
    'MIN_TEMP': 9,
    'SOIL_W1': 10, 
    'SOIL_W3': 11, 
    'SOIL_T1': 12, 
    'DEWPOINT': 13,
    'SOLAR_RAD': 14
}

CSV_PATH = '/kaggle/input/datasets/ashifahmedshuvo/hazardnet/Events/BGD_climatic_hazards_dataset_2000_2026.csv'
TIF_DIR = '/kaggle/input/datasets/ashifahmedshuvo/hazardnet/Tensors' 
OUTPUT_DIR = '/kaggle/working/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, 'hazardnet_with_severity.csv')

OM_CONFIG = {
    "openmeteo_base_url": "https://archive-api.open-meteo.com/v1/archive",
    "openmeteo_daily_vars": [
        "temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
        "precipitation_sum", "wind_speed_10m_max", "relative_humidity_2m_mean",
        "et0_fao_evapotranspiration_sum"
    ],
    "openmeteo_timezone": "Asia/Singapore",
    "request_timeout": 30,
    "rate_limit_delay": 0.5
}

# ==============================================================================
# 2. HYBRID COGNITIVE: TIF-BASED PHYSICAL INDEX FORMULAS
# ==============================================================================
def is_tif_empty(img_array):
    return np.mean(img_array) < 1e-5

def calc_ndvi(nir, red):
    """Normalized Difference Vegetation Index (Corrected: uses NIR[4] and Red[3])"""
    return (nir - red) / (nir + red + 1e-6)
    
def calc_vhi(nir, red, temp):
    """Vegetation Health Index (Drought Severity)"""
    ndvi = calc_ndvi(nir, red)
    # VCI: Vegetation Condition Index
    vci = (ndvi - np.nanmin(ndvi)) / (np.nanmax(ndvi) - np.nanmin(ndvi) + 1e-6)
    # TCI: Temperature Condition Index (Inverted: cooler is healthier)
    tci = (np.nanmax(temp) - temp) / (np.nanmax(temp) - np.nanmin(temp) + 1e-6)
    vhi = 0.5 * vci + 0.5 * tci
    return float(np.clip(1.0 - np.nanmean(vhi), 0.0, 1.0))

def calc_ehf(max_temp, temp_2m):
    """Heat Wave Severity (Excess Heat Factor)"""
    threshold = np.nanpercentile(temp_2m, 90)
    ehf = np.maximum(0, temp_2m - threshold) * np.maximum(0, max_temp - temp_2m)
    return float(np.clip(np.nanmean(ehf), 0.0, 1.0))

def calc_sar_flood(vv, vh, precip):
    """Flood Severity using SAR backscatter and precipitation"""
    sar_ratio = (vv - vh) / (vv + vh + 1e-6)
    severity = (1.0 - np.nanmean(sar_ratio)) * (np.nanmean(precip) / (np.nanmax(precip) + 1e-6))
    return float(np.clip(severity, 0.0, 1.0))

def calc_fire_severity(temp_2m, solar_rad, soil_w1):
    """
    Fire Severity (Corrected: Uses Thermal Stress + Solar Radiation + Soil Dryness 
    as a robust proxy since SWIR is handled correctly at index 5 now).
    """
    # Thermal anomaly (higher temp = higher risk)
    thermal = np.clip((temp_2m - np.nanmean(temp_2m)) / (np.nanmax(temp_2m) - np.nanmean(temp_2m) + 1e-6), 0, 1)
    # Solar desiccation (higher solar rad = higher risk)
    solar = np.clip(solar_rad / (np.nanmax(solar_rad) + 1e-6), 0, 1)
    # Soil dryness (lower soil moisture = higher risk)
    dryness = np.clip(1.0 - (soil_w1 / (np.nanmax(soil_w1) + 1e-6)), 0, 1)
    
    severity = 0.4 * thermal + 0.3 * solar + 0.3 * dryness
    return float(np.clip(np.nanmean(severity), 0.0, 1.0))

def calc_cold_wave_tif(min_temp):
    """Cold Wave Severity (BMD Threshold: ≤16°C)"""
    # Anomaly from 16°C threshold. 10°C drop = 1.0 severity
    anomaly = np.clip((16.0 - min_temp) / 10.0, 0.0, 1.0)
    return float(np.clip(np.nanmean(anomaly), 0.0, 1.0))

# ==============================================================================
# 3. API FETCHERS & HYBRID COGNITIVE VALIDATION
# ==============================================================================

def safe_float(val, default=0.0):
    try:
        f = float(val)
        return default if np.isnan(f) else f
    except Exception:
        return default

# ==============================================================================
# 4. OPEN-METEO FALLBACK FORMULAS (OUTLIER-CAPPED & PHYSICALLY GROUNDED)
# ==============================================================================
def om_calc_severe_storm(precip_max, wind_max):
    p = safe_float(precip_max, 0.0) / 100.0
    w = max(safe_float(wind_max, 0.0) - 50.0, 0.0) / 100.0
    return float(np.clip(0.6 * w + 0.4 * min(p, 1.0), 0.0, 1.0))

def om_calc_cold_wave(temp_min, duration_days):
    cold_anomaly = np.clip((16.0 - safe_float(temp_min, 16.0)) / 10.0, 0.0, 1.0)
    duration_factor = np.clip(safe_float(duration_days, 1.0) / 5.0, 0.0, 1.0)
    return float(np.clip(0.7 * cold_anomaly + 0.3 * duration_factor, 0.0, 1.0))

def om_calc_fire(temp_max, wind_max, et_sum):
    heat = np.clip((safe_float(temp_max, 30.0) - 25.0) / 15.0, 0.0, 1.0)
    wind = np.clip((safe_float(wind_max, 10.0) - 5.0) / 20.0, 0.0, 1.0)
    dryness = np.clip(safe_float(et_sum, 3.0) / 6.0, 0.0, 1.0)
    return float(np.clip(0.4 * heat + 0.3 * wind + 0.3 * dryness, 0.0, 1.0))

def om_calc_tropical_cyclone(wind_max, precip_sum):
    w = max(safe_float(wind_max, 0.0) - 50.0, 0.0) / 150.0
    p = safe_float(precip_sum, 0.0) / 300.0
    return float(np.clip(0.7 * min(w, 1.0) + 0.3 * min(p, 1.0), 0.0, 1.0))

def om_calc_drought(temp_max, precip_sum):
    temp_stress = np.clip((safe_float(temp_max, 25.0) - 25.0) / 20.0, 0.0, 1.0)
    precip_deficit = np.clip((200.0 - safe_float(precip_sum, 200.0)) / 200.0, 0.0, 1.0)
    return float(np.clip(0.6 * temp_stress + 0.4 * precip_deficit, 0.0, 1.0))

def om_calc_flood(precip_sum, precip_max):
    p_factor = np.clip(safe_float(precip_sum, 0.0) / 300.0, 0.0, 1.0)
    i_factor = np.clip(safe_float(precip_max, 0.0) / 100.0, 0.0, 1.0)
    return float(np.clip(0.5 * p_factor + 0.5 * i_factor, 0.0, 1.0))

def om_calc_heat_wave(temp_max, duration_days):
    temp_anomaly = np.clip((safe_float(temp_max, 30.0) - 30.0) / 15.0, 0.0, 1.0)
    duration_factor = np.clip(safe_float(duration_days, 1.0) / 5.0, 0.0, 1.0)
    return float(np.clip(0.7 * temp_anomaly + 0.3 * duration_factor, 0.0, 1.0))

def fetch_openmeteo(lat, lon, start_date, end_date):
    params = {
        "latitude": float(lat), "longitude": float(lon),
        "start_date": start_date, "end_date": end_date,
        "daily": ",".join(OM_CONFIG["openmeteo_daily_vars"]),
        "timezone": OM_CONFIG["openmeteo_timezone"],
    }
    try:
        response = requests.get(OM_CONFIG["openmeteo_base_url"], params=params, timeout=OM_CONFIG["request_timeout"])
        if response.status_code == 200:
            return response.json()
        return None
    except Exception:
        return None

# ==============================================================================
# 5. MAIN PROCESSING LOOP (TIF FIRST)
# ==============================================================================
print(f"Loading CSV from {CSV_PATH}...")
df = pd.read_csv(CSV_PATH)

df = df[df['Hazard_Type'] != 'Earthquake'].copy()
print(f"Filtered out Earthquake events. Remaining events to process: {len(df)}")

severity_scores = []
data_sources = []
confidences = []

print(f"Processing {len(df)} events for HazardNet Severity Indexing (TIF Phase)...")

for idx, row in tqdm(df.iterrows(), total=len(df), desc="TIF Processing"):
    event_id = row['Event_ID_Internal']
    hazard = row['Hazard_Type']

    tif_pattern = os.path.join(TIF_DIR, f"{event_id}_T*_15B.tif")
    tif_files = glob.glob(tif_pattern)

    if not tif_files:
        severity_scores.append(np.nan)
        data_sources.append('Missing_TIF')
        confidences.append(0.5)
        continue

    peak_tif = sorted(tif_files)[-1]

    try:
        with rasterio.open(peak_tif) as src:
            img = src.read()

            if is_tif_empty(img):
                severity_scores.append(np.nan)
                data_sources.append('Empty_TIF')
                confidences.append(0.5)
                continue

            nir = img[BANDS['NIR']]
            red = img[BANDS['Red']]
            temp = img[BANDS['TEMP_2M']]
            max_t = img[BANDS['MAX_TEMP']]
            min_t = img[BANDS['MIN_TEMP']]
            vv = img[BANDS['SAR_VV']]
            vh = img[BANDS['SAR_VH']]
            precip = img[BANDS['PRECIP']]
            solar_rad = img[BANDS['SOLAR_RAD']]
            soil_w1 = img[BANDS['SOIL_W1']]

            if hazard == 'Drought':
                severity_scores.append(calc_vhi(nir, red, temp))
                data_sources.append('TIF_Calculated')
                confidences.append(1.0) # High confidence for direct TIF calculation
            elif hazard == 'Heat Wave':
                severity_scores.append(calc_ehf(max_t, temp))
                data_sources.append('TIF_Calculated')
                confidences.append(1.0)
            elif hazard in ['Flood', 'Flash Flood']:
                severity_scores.append(calc_sar_flood(vv, vh, precip))
                data_sources.append('TIF_Calculated')
                confidences.append(1.0)
            elif hazard == 'Fire':
                severity_scores.append(calc_fire_severity(temp, solar_rad, soil_w1))
                data_sources.append('TIF_Calculated')
                confidences.append(1.0)
            elif hazard == 'Cold Wave':
                severity_scores.append(calc_cold_wave_tif(min_t))
                data_sources.append('TIF_Calculated')
                confidences.append(1.0)
            elif hazard in ['Tropical Cyclone', 'Severe Local Storm']:
                # Proxy using SAR variability + precipitation
                severity_scores.append(float(np.clip(np.nanmean(precip) * np.nanstd(vv), 0.0, 1.0)))
                data_sources.append('TIF_Calculated')
                confidences.append(0.85) # Slightly lower confidence for proxy calculation
            else:
                severity_scores.append(np.nan)
                data_sources.append('Unsupported_Hazard_TIF')
                confidences.append(0.50)

    except Exception as e:
        severity_scores.append(np.nan)
        data_sources.append('TIF_Error')
        confidences.append(0.5)

df['Raw_Severity_Score'] = severity_scores
df['Data_Source'] = data_sources
df['Confidence'] = confidences

# ==============================================================================
# 6. SMART FALLBACK FOR MISSING/FAILED SCORES & SUSPICIOUS ZEROS
# ==============================================================================
failure_mask = df['Data_Source'].isin([
    'Unsupported_Hazard_TIF', 'TIF_Error', 'Empty_TIF', 'Missing_TIF'
])

nan_mask = df['Raw_Severity_Score'].isna()
suspicious_zero_mask = (df['Raw_Severity_Score'] == 0.0)

target_mask = failure_mask | nan_mask | suspicious_zero_mask
df_to_fix = df[target_mask].copy()

if len(df_to_fix) > 0:
    print(f"\n Found {len(df_to_fix)} events requiring API fallback (including suspicious 0.0s). Triggering Open-Meteo...")
    new_scores = []
    new_confidences = []
    
    for idx, row in tqdm(df_to_fix.iterrows(), total=len(df_to_fix), desc="API Fallback"):
        try:
            start_d = pd.to_datetime(row['GEE_Start']).strftime('%Y-%m-%d')
            end_d = pd.to_datetime(row['GEE_End']).strftime('%Y-%m-%d')
            duration = max((pd.to_datetime(end_d) - pd.to_datetime(start_d)).days, 1)
            
            hazard = row['Hazard_Type']
            score = np.nan
            conf = 0.50
            
            
            if hazard == 'Tropical Cyclone':
                api_data = fetch_openmeteo(row['Latitude'], row['Longitude'], start_d, end_d)
                if api_data and "daily" in api_data:
                    daily = api_data["daily"]
                    w_max = np.max(daily.get("wind_speed_10m_max", [0.0]))
                    p_sum = np.sum(daily.get("precipitation_sum", [0.0]))
                    score = om_calc_tropical_cyclone(w_max, p_sum)
                    conf = 0.85 # Strong confidence in Open-Meteo cyclone proxy
                        
            else:
                api_data = fetch_openmeteo(row['Latitude'], row['Longitude'], start_d, end_d)
                if api_data and "daily" in api_data:
                    daily = api_data["daily"]
                    temp_min = np.min(daily.get("temperature_2m_min", [15.0]))
                    temp_max = np.max(daily.get("temperature_2m_max", [30.0]))
                    precip_max = np.max(daily.get("precipitation_sum", [0.0]))
                    precip_sum = np.sum(daily.get("precipitation_sum", [0.0]))
                    wind_max = np.max(daily.get("wind_speed_10m_max", [0.0]))
                    et_sum = np.sum(daily.get("et0_fao_evapotranspiration_sum", [0.0]))
                    
                    if hazard == 'Severe Local Storm':
                        score = om_calc_severe_storm(precip_max, wind_max)
                        conf = 0.85
                    elif hazard == 'Cold Wave':
                        score = om_calc_cold_wave(temp_min, duration)
                        conf = 0.85
                    elif hazard == 'Fire':
                        score = om_calc_fire(temp_max, wind_max, et_sum)
                        conf = 0.85
                    elif hazard == 'Drought':
                        score = om_calc_drought(temp_max, precip_sum)
                        conf = 0.85
                    elif hazard in ['Flood', 'Flash Flood']:
                        score = om_calc_flood(precip_sum, precip_max)
                        conf = 0.85
                    elif hazard == 'Heat Wave':
                        score = om_calc_heat_wave(temp_max, duration)
                        conf = 0.85
            
            new_scores.append(score)
            new_confidences.append(conf)
            
        except Exception:
            new_scores.append(np.nan)
            new_confidences.append(0.50)
            
        time.sleep(OM_CONFIG["rate_limit_delay"])

    df.loc[target_mask, 'Raw_Severity_Score'] = new_scores
    df.loc[target_mask, 'Data_Source'] = np.where(
        df.loc[target_mask, 'Data_Source'].isin(['Missing_TIF', 'Empty_TIF', 'TIF_Error', 'Unsupported_Hazard_TIF']) | suspicious_zero_mask.loc[target_mask],
        'API_Corrected',
        df.loc[target_mask, 'Data_Source']
    )
    df.loc[target_mask, 'Confidence'] = new_confidences

# ==============================================================================
# 7. FINAL NORMALIZATION, UNCERTAINTY BINNING & SAFETY GUARDRAILS
# ==============================================================================
print("\nRe-normalizing per hazard type with variance preservation...")

def normalize_group_safe(group):
    min_val, max_val = group.min(), group.max()
    if pd.isna(min_val) or pd.isna(max_val):
        return pd.Series(np.nan, index=group.index)
    if (max_val - min_val) == 0:
        return group
    return (group - min_val) / (max_val - min_val)

df['Severity_Index'] = df.groupby('Hazard_Type')['Raw_Severity_Score'].transform(normalize_group_safe)

df['Severity_Index'] = df['Severity_Index'].fillna(0.0)
df['Raw_Severity_Score'] = df['Raw_Severity_Score'].fillna(0.0)
df['Data_Source'] = df['Data_Source'].fillna('Default_Fallback')
df['Confidence'] = df['Confidence'].fillna(0.50)

def bin_confidence(conf):
    if conf >= 0.85: return "Certain"
    elif conf >= 0.70: return "Probable"
    else: return "Uncertain"

df['Confidence_Bin'] = df['Confidence'].apply(bin_confidence)
df['Requires_Manual_Review'] = df['Confidence_Bin'] == 'Uncertain'

df.to_csv(OUTPUT_CSV, index=False)
print(f"\n Successfully saved final corrected CSV to: {OUTPUT_CSV}")

print("\n FINAL Severity Index Stats by Hazard Type:")
print(df.groupby('Hazard_Type')['Severity_Index'].agg(['count', 'mean', 'std']).round(3))

print("\n Uncertainty Calibration (Target ECE ≤0.05):")
print(df['Confidence_Bin'].value_counts(normalize=True).round(3))

print("\n Data Source Breakdown:")
print(df['Data_Source'].value_counts())

print(f"\n Safety Guardrail: {df['Requires_Manual_Review'].sum()} events flagged for manual review.")